In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from src.predict import load_model_payload, matchup_probabilities
from src.plots import expected_goals, plot_goal_probability_heatmap

payload = load_model_payload("../models")

if "goal_model" not in payload:
    raise ValueError(
        "This model payload does not include goal_model. "
        "Retrain with: uv run worldcup train"
    )

countries = sorted(payload["latest"].keys())

In [ ]:
team_a = widgets.Dropdown(
    options=countries,
    value="Mexico",
    description="X axis",
    layout=widgets.Layout(width="320px"),
)
team_b = widgets.Dropdown(
    options=countries,
    value="South Africa",
    description="Y axis",
    layout=widgets.Layout(width="320px"),
)
max_goals = widgets.IntSlider(
    value=6,
    min=3,
    max=10,
    step=1,
    description="Max goals",
    continuous_update=False,
    layout=widgets.Layout(width="420px"),
)
plot_button = widgets.Button(description="Plot heatmap", button_style="primary")
output = widgets.Output()


def render_heatmap(_=None):
    with output:
        output.clear_output(wait=True)
        if team_a.value == team_b.value:
            print("Choose two different countries.")
            return

        probabilities = matchup_probabilities(
            payload,
            team_a.value,
            team_b.value,
            tournament="FIFA World Cup",
            neutral=1,
        )
        goals_a, goals_b = expected_goals(
            payload,
            team_a.value,
            team_b.value,
            tournament="FIFA World Cup",
            neutral=1,
        )

        print(f"{team_a.value} win probability: {probabilities[2]:.1%}")
        print(f"Draw probability: {probabilities[1]:.1%}")
        print(f"{team_b.value} win probability: {probabilities[0]:.1%}")
        print(f"{team_a.value} expected goals: {goals_a:.2f}")
        print(f"{team_b.value} expected goals: {goals_b:.2f}")
        print()

        fig, ax, matrix = plot_goal_probability_heatmap(
            payload,
            team_a.value,
            team_b.value,
            max_goals=max_goals.value,
        )
        display(fig)
        plt.close(fig)
        display((matrix * 100).round(2))


plot_button.on_click(render_heatmap)

controls = widgets.VBox(
    [
        widgets.HBox([team_a, team_b]),
        widgets.HBox([max_goals, plot_button]),
    ]
)
display(controls, output)
render_heatmap()